# Extracción del calendario laboral Madrid 2022

**Fuente:** [Portal de datos abiertos del Ayuntamiento de Madrid](https://datos.madrid.es/portal/site/egob/menuitem.c05c1f754a33a9fbe4b2e4b284f1a5a0/?vgnextoid=9f710c96da3f9510VgnVCM2000001f4a900aRCRD&vgnextchannel=374512b9ace9f310VgnVCM100000171f5a0aRCRD&vgnextfmt=default)

In [2]:
import pandas as pd

# Cargar calendario laboral 2022
df_calendario = pd.read_csv('calendario.csv', sep=';', encoding='latin1')

print("Dimensiones:", df_calendario.shape)
print("Columnas:", list(df_calendario.columns))
print("\nPrimeras 10 filas:")
df_calendario.head(10)

Dimensiones: (4747, 5)
Columnas: ['ï»¿Dia', 'Dia_semana', 'laborable / festivo / domingo festivo', 'Tipo de Festivo', 'Festividad']

Primeras 10 filas:


,ï»¿Dia,Dia_semana,laborable / festivo / domingo festivo,Tipo de Festivo,Festividad
0,01/01/2013,martes,festivo,Festivo nacional,AÃ±o Nuevo
1,02/01/2013,miercoles,laborable,NaN,NaN
2,03/01/2013,jueves,laborable,NaN,NaN
3,04/01/2013,viernes,laborable,NaN,NaN
4,05/01/2013,sabado,sabado,NaN,NaN
5,06/01/2013,domingo,domingo,NaN,NaN
6,07/01/2013,lunes,festivo,Festivo nacional,Traslado de la Epifania del SeÃ±or
7,08/01/2013,martes,laborable,NaN,NaN
8,09/01/2013,miercoles,laborable,NaN,NaN
9,10/01/2013,jueves,laborable,NaN,NaN


### Estructura del Dataset de Calendario Laboral

El dataset contiene información sobre días laborables, festivos y fines de semana en Madrid durante 2022. 

**Campos:**
- **fecha**: Fecha en formato YYYY-MM-DD
- **dia_semana**: Nombre del día de la semana
- **laborable / festivo / domingo festivo**: Tipo de día (laborable, sábado, domingo, festivo)
- **Tipo de festivo**, **Festividad**

## Transformaciones
+ Eliminaremos las columnas `Tipo de festivo`,`Festividad`. No son relevantes en nuestro análisis.
+ Tipado de datos correcto
+ 

In [3]:
# Eliminar columnas no relevantes, renombrar
df_calendario = df_calendario.drop(columns=['Tipo de Festivo', 'Festividad'])
df_calendario = df_calendario.rename(columns={'ï»¿Dia': 'Dia'})
df_calendario = df_calendario.rename(columns={'laborable / festivo / domingo festivo': 'Tipo'})

# Tipado de datos
df_calendario['Dia'] = pd.to_datetime(df_calendario['Dia'], format='%d/%m/%Y')
df_calendario['Dia_semana'] = df_calendario['Dia_semana'].astype('category')
df_calendario['Tipo'] = df_calendario['Tipo'].astype('category')


print("Dataset transformado:")
print("Dimensiones:", df_calendario.shape)
print("\nTipos:")
print(df_calendario.dtypes)
print("\nPrimeras filas:")
df_calendario.head()

Dataset transformado:
Dimensiones: (4747, 3)

Tipos:
Dia           datetime64[ns]
Dia_semana          category
Tipo                category
dtype: object

Primeras filas:


,Dia,Dia_semana,Tipo
0,2013-01-01,martes,festivo
1,2013-01-02,miercoles,laborable
2,2013-01-03,jueves,laborable
3,2013-01-04,viernes,laborable
4,2013-01-05,sabado,sabado


### Filtro: Solo 2022

Nos centramos en 2022 para alinear el calendario con el resto de datasets del proyecto.

In [4]:
# Filtrar a 2022
mask_2022 = df_calendario['Dia'].dt.year == 2022
df_calendario = df_calendario[mask_2022].copy()
print('Filas tras filtrar 2022:', len(df_calendario))
df_calendario.head()

Filas tras filtrar 2022: 365


,Dia,Dia_semana,Tipo
3286,2022-01-01,sabado,Festivo
3287,2022-01-02,domingo,NaN
3288,2022-01-03,lunes,NaN
3289,2022-01-04,martes,NaN
3290,2022-01-05,miercoles,NaN


### Rellenado de la columna `Tipo`

En la columna Tipo, tenemos solo valores 'Festivo'. Nos interesa tambien diferenciar entre 'laborable', 'sabado y 'domingo'.

In [ ]:
# Rellenar 'Tipo' cuando esté vacío usando 'Dia_semana'
import pandas as pd

# Asegurar tipo objeto para poder asignar nuevas categorías
df_calendario['Tipo'] = df_calendario['Tipo'].astype(object)

mascara_vacia = df_calendario['Tipo'].isna() | (df_calendario['Tipo'].astype(str).str.strip() == '')
dia_semana = df_calendario['Dia_semana'].astype(str).str.lower()

mapa = {
    'sabado': 'sabado',
    'sábado': 'sabado',
    'domingo': 'domingo',
    'lunes': 'laborable',
    'martes': 'laborable',
    'miercoles': 'laborable',
    'miércoles': 'laborable',
    'jueves': 'laborable',
    'viernes': 'laborable'
}

# Asignar solo donde 'Tipo' esté vacío
df_calendario.loc[mascara_vacia, 'Tipo'] = dia_semana.map(mapa)

# Devolver a categoría
df_calendario['Tipo'] = df_calendario['Tipo'].astype('category')

print("Resumen de 'Tipo' tras rellenar:")
print(df_calendario['Tipo'].value_counts(dropna=False))

### Guardado del calendario transformado en un archivo CSV


In [20]:
# Guardar calendario transformado
ruta_salida = 'calendario_clean.csv'
df_calendario.to_csv(ruta_salida, sep=';', encoding='latin1', index=False)
print(f"Archivo guardado: {ruta_salida} | Filas: {len(df_calendario)} | Columnas: {df_calendario.shape[1]}")

Archivo guardado: calendario_clean.csv | Filas: 4747 | Columnas: 3
